In [1]:
import pandas as pd
import numpy as np
from scipy.integrate import solve_ivp
from scipy.optimize import minimize, LinearConstraint, Bounds
import random
from tqdm import tqdm
from joblib import Parallel, delayed

In [ ]:
filepath = ""
df = pd.read_csv(f"{filepath}Normalized Stopped-flow Gle1-Dbp5 RNA binding data adjusted for bleaching.csv")

In [3]:
atp_values = [1.0, 2.5, 5.0, 10.0, 50.0]    
t_span = (0, 100)
t_eval = np.geomspace(1e-3, 100, 4000)

def reaction_equations(t, y, kf1, kr1, kf2, kr2, kf3, kr3, kf4, kr4, kf5, kr5, kf6, kr6, kf7, kr7, kf8, kr8, kf9, kr9, kf10, kr10, kf11, kr11):

    # state variables
    H, T, HT, R, HRT, HRDP, HRD, P, HR, HDP, HD, D = y

    # reaction fluxes
    v1  = kf1 * H * T - kr1 * HT
    v2  = kf2 * HT * R - kr2 * HRT
    v3  = kf3 * HRT - kr3 * HRDP
    v4  = kf4 * HRDP - kr4 * HRD * P
    v5  = kf5 * HRD - kr5 * HR * D
    v6  = kf6 * HR - kr6 * H * R
    v7  = kf7 * HT - kr7 * HDP
    v8  = kf8 * HDP - kr8 * HD * P
    v9  = kf9 * HD - kr9 * H * D
    v10 = kf10 * HDP * R - kr10 * HRDP
    v11 = kf11 * HRD - kr11 * HD * R
    
 
    # differential equations
    dH    = -v1 + v6 + v9
    dT    = -v1 
    dHT   = v1 - v2 - v7
    dR    = -v2 + v6 - v10 + v11
    dHRT  = v2 - v3
    dHRDP = v3 - v4 + v10
    dHRD  = v4 - v5 - v11
    dP    = v4 + v8
    dHR   = v5 - v6
    dHDP  = v7 - v8 - v10
    dHD   = v8 - v9 + v11
    dD    = v5 + v9
    

    return np.array([dH, dT, dHT, dR, dHRT, dHRDP, dHRD, dP, dHR, dHDP, dHD, dD])

In [ ]:
def global_objective_function(free_params, locked_indices, locked_values, 
                              t_exp, df_exp, base_y0, t_span, atp_values, rtol, atol):
    """
    Reconstructs the full parameter set and runs the simulation.
    """
    # reconstructs the full test_rates array
    total_params_count = len(free_params) + len(locked_indices)
    test_rates = np.zeros(total_params_count)
    
    # identifies indices that are NOT locked
    free_indices = [i for i in range(total_params_count) if i not in locked_indices]
    
    # maps values back to their correct positions
    test_rates[free_indices] = free_params
    test_rates[locked_indices] = locked_values

    all_residuals = []
    
    for atp_idx, atp_val in enumerate(atp_values):
        y0_current = base_y0.copy()
        y0_current[1] = atp_val 
        
        y_exp = df_exp.iloc[:, atp_idx + 1].to_numpy()
        
        # runs simulation with the reconstructed test_rates
        sol = solve_ivp(
            reaction_equations, 
            t_span, 
            y0_current, 
            t_eval=t_exp, 
            method='BDF',
            rtol=rtol,
            atol=atol,  
            args=test_rates
        )
        
        y_sim = sol.y[3]
        mask = t_exp > 1e-2
        
        all_residuals.append(y_sim[mask] - y_exp[mask])
        
    return np.sum(np.concatenate(all_residuals)**2)


def least_squares_fitting(search_range, rtol=1e-5, atol=1e-7, ftol=1e-8, max_iter = 300, initial = None, lock=None):
    """
    Returns the least squares fit between the data and simulations from a single initialization
    """
    if lock is None:
        lock = []

    # prepares initial guesses and bounds
    all_lower = [r[0] for r in search_range]
    all_upper = [r[1] for r in search_range]

    if initial is not None:
        all_initial = initial
    else:
        all_initial = [random.uniform(l, h) for l, h in zip(all_lower, all_upper)]

    # separates "Free" parameters from "Locked" parameters
    free_initial = [val for i, val in enumerate(all_initial) if i not in lock]
    
    # formatted specifically for `minimize` using the Bounds object
    free_lower = [val for i, val in enumerate(all_lower) if i not in lock]
    free_upper = [val for i, val in enumerate(all_upper) if i not in lock]
    free_bounds = Bounds(free_lower, free_upper)
    
    # stores the values of the locked parameters to pass into the objective
    locked_values = [all_initial[i] for i in lock]

    base_y0 = [5, 0, 0, 0.5, 0, 0, 0, 0, 0, 0, 0, 0] 
    t_exp = df.iloc[:, 0].to_numpy()

    num_params = len(free_initial)
    lag_phase_sum = np.zeros(num_params)

    # adds lagphase constraint
    lag_phase_sum[4] = 1.0 
    lag_phase_sum[5] = 1.0 
    lag_phase_sum[6] = 1.0 

    sum_constraint = LinearConstraint(lag_phase_sum, lb=11.1, ub=13.3)

    ratio_arr = np.zeros(num_params)
    ratio_arr[4] = -2.0  
    ratio_arr[5] = 1.0
    ratio_arr[6] = 1.0   

    # adds burst phase constraint
    ratio_constraint = LinearConstraint(ratio_arr, lb=0.0, ub=np.inf)

    res = minimize(
        global_objective_function, 
        free_initial, 
        args=(lock, locked_values, t_exp, df, base_y0, t_span, atp_values, rtol, atol), 
        method='SLSQP',
        bounds=free_bounds,      
        constraints=[sum_constraint, ratio_constraint], 
        options={'ftol': ftol, 'maxiter': max_iter,'disp': False}
    )

    final_rates = np.zeros(len(search_range))
    free_indices = [i for i in range(len(search_range)) if i not in lock]
    final_rates[free_indices] = res.x
    final_rates[lock] = locked_values
    
    res.full_x = final_rates
    
    return res
    

In [ ]:
fit_ranges = []
sweep_ranges = []
kinetic_rate_constants = []

# H + T <=> HT
fit_ranges.extend([(0.1, 0.7), (0.1, 7)])
sweep_ranges.extend([(0.0, 0.4), (0.0, 6.0)])
kinetic_rate_constants.extend([r"k+T,G", r"k-T,G"])

# HT + R <=> HRT
fit_ranges.extend([(5.3, 9.3), (1.9, 3.9)])
sweep_ranges.extend([(0.0, 15.0), (0.0, 20.0)])
kinetic_rate_constants.extend([r"k+R,GT", r"k-R,GT"])

# HRT <=> HRDP # forward and reverse (for sum)
fit_ranges.extend([(2.0, 13.3), (0, 13.3)])
sweep_ranges.extend([(1.0, 13.3), (0, 13.3)])
kinetic_rate_constants.extend([r"k+h,RG", r"k-h,RG"])

# HRDP <=> HRD + P  # and forward phosphate release (for sum) sums to 12.2 +/- 1.1
fit_ranges.extend([(2.0, 13.3), (0.0, 0.01)])
sweep_ranges.extend([(1.0, 13.3), (0.0, 25.0)])
kinetic_rate_constants.extend([r"k-Pi,RGD", r"k+Pi,RGD"])

# HRD <=> HR + D
fit_ranges.extend([(0.0, 100.0), (0.0, 0.01)])   
sweep_ranges.extend([(0.0, 100.0), (0.0, 100.0)]) 
kinetic_rate_constants.extend([r"k-D,RG", r"k+D,RG"])   

# HR <=> H + R
fit_ranges.extend([(100.0, 101.0), (0.0, 0.01)])
sweep_ranges.extend([(0.0, 16.0), (0.0, 2.0)])
kinetic_rate_constants.extend([r"k-R,G", r"k+R,G"])

# HT <=> HDP
fit_ranges.extend([(0.4, 0.8), (0.1, 2)]) 
sweep_ranges.extend([(0.0, 2.0), (0.0, 6.0)]) 
kinetic_rate_constants.extend([r"k+h,G", r"k-h,G"])   

# HDP <=> HD + P
fit_ranges.extend([(0.2, 0.6), (0.0, 0.01)])  
sweep_ranges.extend([(0.0, 6.0), (0.0, 100.0)])
kinetic_rate_constants.extend([r"k-Pi,GD", r"k+Pi,GD"])  

# HD <=> H + D
fit_ranges.extend([(100.0, 101.0), (0.0, 0.01)])
sweep_ranges.extend([(0.0, 6.0), (0.0, 6.0)])
kinetic_rate_constants.extend([r"k-D,G", r"k+D,G"])

# HDP + R <==> HRDP
fit_ranges.extend([(0.0, 100.0), (0.0, 100.0)])
sweep_ranges.extend([(0.0, 100.0), (0, 100.0)])
kinetic_rate_constants.extend([r"k+R,GDP", r"k-R,GDP"])

# HRD <==> HD + R
fit_ranges.extend([(3.0, 7.4), (0.6, 0.8)])
sweep_ranges.extend([(0.0, 16.0), (0.0, 16.0)])
kinetic_rate_constants.extend([r"k-R,GD", r"k+R,GD"])


In [ ]:
def score_sim_curve_chi2(kinetic_parameters, t_exp, y_exp, atp_value):
    """
    Returns the chi2 of a single curve
    """
    
    y0 = [5, atp_value, 0, 0.5, 0, 0, 0, 0, 0, 0, 0, 0] 
    sol = solve_ivp(reaction_equations, t_span, y0, t_eval=t_exp, method='BDF', args=kinetic_parameters)
    y_sim = sol.y[3] 
    mask = t_exp > 1e-2
    residuals = y_exp[mask] - y_sim[mask]
    
    return np.sum(residuals**2)


def score_multiple_curves_chi2(kinetic_parameters):
    """
    Returns global chi2 of all curves
    """

    global_chi_squared = 0
    for i, atp_conc in enumerate(atp_values):
        t_exp = df.iloc[:, 0].to_numpy()
        y_exp = df.iloc[:, i + 1].to_numpy()
        curve_chi2 = score_sim_curve_chi2(
            kinetic_parameters=kinetic_parameters, 
            t_exp=t_exp, 
            y_exp=y_exp, 
            atp_value=atp_conc
        )
        global_chi_squared += curve_chi2

    return global_chi_squared

In [ ]:
def _run_single_fit(fit_ranges):
    """
    Input function for random initialization
    """

    random_fit = least_squares_fitting(fit_ranges, max_iter=300)
    random_fit_parameters = np.round(random_fit.full_x, 2)
    
    chi2 = score_multiple_curves_chi2(random_fit_parameters)
        
    return {
        'Parameters': random_fit_parameters.tolist(),
        'Chi_Squared': chi2,
    }

def random_initialization(num_iterations, fit_ranges, n_jobs=-1):
    """
    Returns a DataFrame of fits from randomly initialized fits within parameter bounds and returns the best fit
    """

    # queues up the parallel ta sks and tell joblib to return them as a generator
    parallel_tasks = Parallel(n_jobs=n_jobs, return_as="generator")(
        delayed(_run_single_fit)(fit_ranges) 
        for _ in range(num_iterations)
    )
    
    # wraps the executing generator in tqdm. 
    # this guarantees the bar only moves forward when a fit ACTUALLY finishes.
    results_list = list(tqdm(parallel_tasks, total=num_iterations, desc="Computing Fits"))
            
    # builds the Output DataFrame
    results_df = pd.DataFrame(results_list)
    results_df = results_df.sort_values(by='Chi_Squared')
    best_fit = results_df['Parameters'].iloc[0]
    
    return results_df, best_fit

In [ ]:
all_fits, best_fit = random_initialization(num_iterations = 1000, fit_ranges=fit_ranges)

Computing Fits: 100%|██████████| 10/10 [44:14<00:00, 265.48s/it]  


In [ ]:
def calculate_single_line(i, fit_ranges, kinetic_parameters, i_lims, grid_size, chi2_min):
    """
    Returns the one dimensional analysis of a single parameter
    """

    # 1D array for the parameter sweep
    X = np.linspace(i_lims[0], i_lims[1], grid_size)
    Z = np.zeros_like(X)
    
    # determines optimal starting value, clipping it to the i_lims bounds if it falls outside
    opt_val = np.clip(kinetic_parameters[i], i_lims[0], i_lims[1])
    
    # find the index in X that is closest to this optimal starting value
    start_idx = np.argmin(np.abs(X - opt_val))
    
    # keeps a copy of the optimal parameters to spawn both directions
    best_fit_params = kinetic_parameters.copy()
    
    # --- Sweep Right (from start_idx to the end) ---
    test_params = best_fit_params.copy()
    for row in range(start_idx, grid_size):
        test_params[i] = X[row]
        
        # sets up the fit ranges, locking only parameter i
        fit_ranges_w_test_params = fit_ranges.copy()
        fit_ranges_w_test_params[i] = (test_params[i], test_params[i] + 0.0001)
        
        # refits with only ONE locked parameter
        refit_parameters = least_squares_fitting(
            fit_ranges_w_test_params, 
            initial=test_params, 
            lock=[i], 
            rtol=1e-5, atol=1e-7, ftol=1e-8, max_iter=50
        )
        refit_parameters = np.round(refit_parameters.full_x, 2)
        
        # scores the refitted curve
        chi2_test = score_multiple_curves_chi2(refit_parameters)
        
        # carries over the optimized parameters for the next fit
        test_params = refit_parameters.copy()
        
        Z[row] = chi2_min / chi2_test

    # --- Sweep Left (from start_idx - 1 down to 0) ---
    test_params = best_fit_params.copy()  # resets to optimal center
    for row in range(start_idx - 1, -1, -1):
        test_params[i] = X[row]
        
        # sets up the fit ranges, locking only parameter i
        fit_ranges_w_test_params = fit_ranges.copy()
        fit_ranges_w_test_params[i] = (test_params[i], test_params[i] + 0.0001)
        
        # refits with only ONE locked parameter
        refit_parameters = least_squares_fitting(
            fit_ranges_w_test_params, 
            initial=test_params, 
            lock=[i], 
            rtol=1e-5, atol=1e-7, ftol=1e-8, max_iter=50
        )
        refit_parameters = np.round(refit_parameters.full_x, 2)
        
        # scores the refitted curve
        chi2_test = score_multiple_curves_chi2(refit_parameters)
        
        # carries over the optimized parameters for the next fit
        test_params = refit_parameters.copy()
        
        Z[row] = chi2_min / chi2_test
            
    return i, X, Z

In [ ]:
def chi2_1D_sweep(kinetic_parameters, search_ranges, grid_size=10):
    """
    Returns the least squares fit between the data and simulations from a single initialization
    """

    chi2_min = score_multiple_curves_chi2(kinetic_parameters)
    num_params = len(kinetic_parameters)
    
    line_data = {}

    print(f"Starting PARALLEL 1D line calculations using Joblib. Resolution: {grid_size} points...")

    tasks = (
        delayed(calculate_single_line)(
            i=param, fit_ranges=fit_ranges, kinetic_parameters=kinetic_parameters, i_lims=search_ranges[param], grid_size=grid_size, chi2_min=chi2_min
        )
        for param in range(num_params)
    )

    results = Parallel(n_jobs=-1, return_as="generator")(tasks)

    for i, X, Z in tqdm(results, total=num_params, desc="Mapping 1D Profiles"):
        line_data[i] = (X, Z)

    return line_data

In [ ]:
line_data = chi2_1D_sweep(kinetic_parameters=best_fit, search_ranges=sweep_ranges, grid_size = 500)

Starting PARALLEL 1D line calculations using Joblib. Resolution: 10 points...


Mapping 1D Profiles: 100%|██████████| 22/22 [22:37<00:00, 61.72s/it]  


In [ ]:
export_path = ""
one_d_plots = []
for i, plot in enumerate(range(len(line_data))):
    plot_name = kinetic_rate_constants[i]
    plot_df = pd.DataFrame({
        'Parameter_Value': line_data[i][0],
        'Chi2_Ratio': line_data[i][1]
    })
    one_d_plots.append((plot_name, plot_df))
for plot_name, plot_df in one_d_plots:
    plot_df.to_csv(f"{export_path}Chi2_1D_profile_{plot_name}.csv", index=False)

In [ ]:
# forces RNA release parameters from Gle1-Dbp5-RNA-ADP to be 0
sum_1d_params = best_fit.copy()
sum_1d_params[8] = 6
sum_1d_params[-2] = 0.0
sum_1d_params[-1] = 0.0
sum_1d_rate_ranges = fit_ranges.copy()
sum_1d_rate_ranges[-2] = (0.0, 0.001)
sum_1d_rate_ranges[-1] = (0.0, 0.001)

# calculating the the one dimesional chi analysis for the sum of the RNA and ADP release from Gle1-Dbp5-RNA-ADP
sum_1d_chi = calculate_single_line(i = 8, fit_ranges = sum_1d_rate_ranges, kinetic_parameters = sum_1d_params, i_lims = (0,8), grid_size = 500, chi2_min = score_multiple_curves_chi2(best_fit))

In [ ]:
export_path = ""
release_sum_df = pd.DataFrame({
    "Parameter_Value" : sum_1d_chi[1],
    "Chi2_Ratio" : sum_1d_chi[2]})
release_sum_df.to_csv(f"{export_path}Chi2_1D_profile_sum_of_product_release.csv", index=False)